# Notebook 3 — ToM-SLM (Small Teacher)

**What happens here:**
A *small* Teacher model (`Qwen2.5-3B-Instruct`) reads the student's first answer
and builds a Theory-of-Mind diagnosis: it infers the question's intent,
identifies what the student misunderstood, and provides targeted guidance.
The Student then revises using that guidance — **without ever being shown the answer**.

```
Code + Question  →  [Student]  →  Draft Answer
                                        ↓
                              [Teacher-SLM (3B)]  →  ToM Diagnosis
                                                          (intent · gap · guidance)
                                                              ↓
                                              [Student]  →  Revised Answer  →  [Judge]  →  Score
```

> *Proposed ToM intervention — Maha Zainab, BEA 2026.*

# BEA 2026 Tutorial — Machine ToM in Source Code Comprehension

> **Maha Zainab · Auburn University · BEA Workshop, ACL 2026**

This notebook is part of a series of four Colab notebooks that walk through the four experimental conditions:

| Notebook | Condition | Description |
|---|---|---|
| 1 | **No-Intervention** | Student answers with few-shot prompting only |
| 2 | **Self-Refine** | Student critiques and revises its own answer |
| 3 | **ToM-SLM** | Small Teacher (3B) diagnoses the gap; Student revises |
| 4 | **ToM-LLM** | Large Teacher (7B) diagnoses the gap; Student revises |


## ⚙️ Setup

Install dependencies and authenticate with Hugging Face.

> **Runtime:** Use a GPU runtime (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install -q transformers accelerate

In [ ]:
# Authenticate with Hugging Face to access gated models (Llama-3.2)
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
import gc, json, textwrap, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device count   : {torch.cuda.device_count()}")

## 📄 Demo Example

In [ ]:
# ── Demo example 
# Source: real RQ1 judged output, q278 (CodeQA, Why).
EXAMPLE = {
    "id":             "q278",
    "dataset":        "CodeQA",
    "category":       "Why",
    "code": (
        "def auto_reconnect_connection(func):\n"
        "    @wraps(func)\n"
        "    def inner(self, *args, **kwargs):\n"
        "        try:\n"
        "            return func(self, *args, **kwargs)\n"
        "        except Exception as e:\n"
        "            if not can_reconnect(e):\n"
        "                raise\n"
        "            self.close()\n"
        "            self.reconnect = True\n"
        "            return func(self, *args, **kwargs)\n"
        "    return inner"
    ),
    "question":         "Why do the bouncer disconnect the client?",
    "reference_answer": "Due to a timeout / etc.",
}

code      = EXAMPLE["code"]
question  = EXAMPLE["question"]
reference = EXAMPLE["reference_answer"]

print("Dataset  :", EXAMPLE["dataset"])
print("Category :", EXAMPLE["category"])
print("Question :", question)
print("Reference:", reference)
print("\nCode:\n")
print(code)

## 🛠️ Model Utilities

Helpers to load, run, and offload models.

In [ ]:
def load_model(model_id, label):
    """Load tokenizer + model onto GPU."""
    print(f"[{label}] Loading tokenizer…")
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    print(f"[{label}] Loading model weights…")
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype="auto", device_map="auto", trust_remote_code=True
    )
    mdl.eval()
    print(f"[{label}] Ready on {mdl.device}")
    return tok, mdl

def offload(model, label):
    """Move model to CPU and free VRAM."""
    model.cpu()
    torch.cuda.empty_cache()
    print(f"[{label}] Offloaded – VRAM freed.")

def restore(model, label):
    model.cuda()
    print(f"[{label}] Restored to GPU.")

def run_model(model, tokenizer, system_prompt, user_prompt, max_new_tokens):
    """Format messages with the chat template and generate a response."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        formatted = (
            f"### System:\n{system_prompt}\n\n"
            f"### User:\n{user_prompt}\n\n### Assistant:\n"
        )
    inputs = tokenizer(
        formatted, return_tensors="pt", padding=True,
        truncation=True, max_length=4096,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 🏛️ Judge

The Judge (`Qwen2.5-Coder-7B-Instruct`) scores the answer on four dimensions (1–5 each):
**Accuracy · Completeness · Clarity · Relevance**.


In [ ]:
JUDGE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_TOKENS_JUDGE = 128

JUDGE_SYSTEM = """You are an expert software engineer and code evaluator with deep experience assessing the quality of answers to source code comprehension questions.

You will be given a code snippet, a question about that code, a reference answer, and a predicted answer.
Use the reference answer as a gold-standard anchor — semantic equivalence to the reference counts as correct.
Your task is to evaluate the predicted answer across four dimensions: Accuracy, Completeness, Clarity, and Relevance.

Dataset context:
The questions come from CodeQA, a free-form question-answering benchmark built from real Python code on GitHub.
Correct answers in CodeQA are typically concise — often a short phrase or a single sentence, not a paragraph.

Scoring dimensions (score each independently on a 1 to 5 integer scale):

ACCURACY — Does the predicted answer correctly reflect what the code actually does?
  5: Completely correct — fully consistent with the code's actual behavior
  4: Mostly correct — minor factual slip that does not change the core meaning
  3: Partially correct — captures something true but misses or misstates a key detail
  2: Mostly incorrect — contains a relevant element but dominated by factual errors
  1: Completely wrong — contradicts the code or addresses something entirely different

COMPLETENESS — Does the predicted answer cover everything the question asks for?
  5: Fully complete — addresses everything the question asks at the right level of detail
  4: Mostly complete — minor omission that does not significantly affect the answer
  3: Partially complete — addresses part of the question but misses an important aspect
  2: Mostly incomplete — only a surface fragment of what is required is present
  1: Entirely incomplete — fails to address the question in any meaningful way

CLARITY — How clearly does the predicted answer communicate its point?
  5: Perfectly clear — unambiguous and easy to understand
  4: Mostly clear — minor phrasing awkwardness that does not impede understanding
  3: Somewhat clear — understandable with effort but awkwardly expressed
  2: Unclear — confusing or ambiguous to the point of impeding understanding
  1: Incomprehensible — incoherent, self-contradictory, or unreadable

RELEVANCE — Does the predicted answer directly target what the question is asking?
  5: Fully relevant — directly and precisely answers the question asked
  4: Mostly relevant — minor tangent that does not distract from the answer
  3: Partially relevant — addresses a related but different aspect of the code
  2: Mostly irrelevant — misses the main point of the question
  1: Completely irrelevant — does not address the question at all

IMPORTANT: Semantic equivalence to the reference answer counts as correct.
IMPORTANT: Score each dimension INDEPENDENTLY.
IMPORTANT: A short answer is NOT incomplete if it fully addresses the question.

CRITICAL: DO NOT PENALIZE FOR PHRASING OR VOCABULARY DIFFERENCES IF THE MEANING IS CORRECT.
CRITICAL: DO NOT PRODUCE ANY TEXT OUTSIDE THE JSON OBJECT — no explanation, no preamble, no markdown.

Respond ONLY with a valid JSON object in exactly this format:
{
  \"accuracy\":     {\"score\": <1-5>},
  \"completeness\": {\"score\": <1-5>},
  \"clarity\":      {\"score\": <1-5>},
  \"relevance\":    {\"score\": <1-5>}
}"""

def build_judge_user(code, question, reference, prediction):
    return (
        f"Code:\n{code}\n\n"
        f"Question:\n{question}\n\n"
        f"Reference Answer:\n{reference}\n\n"
        f"Predicted Answer:\n{prediction}"
    )

def run_judge(model, tokenizer, code, question, reference, prediction):
    user_prompt = build_judge_user(code, question, reference, prediction)
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user",   "content": user_prompt},
    ]
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        formatted = (
            f"### System:\n{JUDGE_SYSTEM}\n\n"
            f"### User:\n{user_prompt}\n\n### Assistant:\n"
        )
    inputs = tokenizer(
        formatted, return_tensors="pt", padding=True,
        truncation=True, max_length=8192,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=MAX_TOKENS_JUDGE,
            do_sample=False, use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    try:
        start  = response.find("{")
        end    = response.rfind("}") + 1
        parsed = json.loads(response[start:end])
    except (json.JSONDecodeError, ValueError):
        print(f"Warning: could not parse judge response: {response[:200]}")
        return {"accuracy": None, "completeness": None, "clarity": None, "relevance": None}
    out = {}
    for key in ("accuracy", "completeness", "clarity", "relevance"):
        details = parsed.get(key, {})
        score   = details.get("score") if isinstance(details, dict) else details
        out[key] = score if isinstance(score, int) and 1 <= score <= 5 else None
    return out

def show_scores(scores, label):
    total = sum(v for v in scores.values() if v is not None)
    print(f"[{label}]  Accuracy={scores['accuracy']}/5  "
          f"Completeness={scores['completeness']}/5  "
          f"Clarity={scores['clarity']}/5  "
          f"Relevance={scores['relevance']}/5  → Total={total}/20")

## 🧠 ToM-SLM Pipeline

Four steps: *draft → small Teacher diagnosis → student revision → judge scoring*.

In [ ]:
# ── ToM-SLM Prompts 

STUDENT_MODEL_ID      = "meta-llama/Llama-3.2-3B-Instruct"
TEACHER_SLM_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
MAX_TOKENS_FIRST    = 128
MAX_TOKENS_TEACHER  = 320
MAX_TOKENS_REVISED  = 128

FEW_SHOT_EXAMPLES = [
    {
        "code":     "def get_name ( self ) : return self . _name",
        "question": "What does the code return?",
        "answer":   "the name",
    },
    {
        "code":     "def is_empty ( self ) : return len ( self . items ) == 0",
        "question": "Does the code check if the list is empty?",
        "answer":   "Yes",
    },
    {
        "code":     "def sort_list ( items ) : return sorted ( items , reverse = True )",
        "question": "How does the code sort the items?",
        "answer":   "in descending order",
    },
]

NOINTV_SYSTEM = (
    "You are an expert software engineer with deep experience in source code comprehension, "
    "code review, and software documentation across multiple programming languages including Python.\n"
    "You will be given a code snippet and a natural language question about that code.\n\n"
    "Your task:\n"
    "- Read the code carefully before answering.\n"
    "- Answer the question directly and concisely based solely on what the code does.\n"
    "- Match the style and length of the examples provided — a short phrase or single sentence is expected.\n"
    "- For Yes/No questions, answer with Yes or No followed by a brief reason only if necessary.\n\n"
    "IMPORTANT: Study the examples carefully before answering.\n"
    "IMPORTANT: The examples define the expected answer style, format, and length — match them exactly.\n"
    "IMPORTANT: If the question asks WHAT the code returns, answer with the thing that is returned, not a description of how it works.\n\n"
    "CRITICAL: DO NOT REPEAT OR PARAPHRASE THE QUESTION IN YOUR ANSWER.\n"
    "CRITICAL: DO NOT ADD EXPLANATIONS OR INFORMATION NOT PRESENT IN THE CODE.\n"
    "CRITICAL: DO NOT PRODUCE A PARAGRAPH WHEN A PHRASE IS SUFFICIENT.\n"
    "CRITICAL: DO NOT HALLUCINATE — base your answer ONLY on the provided code snippet."
)

# The ToM Teacher prompt — infers intent, diagnoses the gap, provides guidance.
# Proposed ToM intervention (Maha Zainab, BEA 2026).
TEACHER_SYSTEM = (
    "You are an expert tutor coaching a student who is answering a source code comprehension question.\n"
    "You will be given a code snippet, a question, and the student's first answer.\n"
    "There is NO reference answer. Reason about the code yourself.\n\n"
    "Respond using EXACTLY these three labeled sections and nothing else:\n\n"
    "QUESTION INTENT: what the question is really asking for and what kind of answer would satisfy it "
    "(e.g. WHAT is returned vs HOW it works vs WHY it exists).\n"
    "STUDENT UNDERSTANDING: what the student's first answer reveals — what they got right and what they missed.\n"
    "GUIDANCE: specific, targeted hints pointing the student to the exact part of the code "
    "they should focus on and how their answer should change.\n\n"
    "CRITICAL: DO NOT WRITE THE STUDENT'S FINAL ANSWER — diagnose and direct only.\n"
    "CRITICAL: USE EXACTLY THE THREE LABELED SECTIONS — no preamble, no extra text, no markdown."
)

STUDENT_REVISED_SYSTEM = (
    "You are an expert software engineer answering a source code comprehension question.\n"
    "A tutor has reviewed your first answer and provided an analysis with targeted guidance.\n"
    "Use the tutor's guidance to write an improved answer.\n\n"
    "IMPORTANT: Keep the answer concise — a short phrase or single sentence.\n"
    "CRITICAL: BASE YOUR ANSWER ONLY ON THE PROVIDED CODE AND THE TUTOR'S GUIDANCE.\n"
    "CRITICAL: OUTPUT ONLY THE IMPROVED ANSWER — no preamble, no explanation, no markdown."
)

def build_nointv_user(code, question):
    lines = []
    for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
        lines.append(f"--- Example {i} ---")
        lines.append(f"Code:\n{ex['code']}")
        lines.append(f"Question: {ex['question']}")
        lines.append(f"Answer: {ex['answer']}")
    lines.append("--- End of Examples ---\n")
    lines.append("Now answer the following question in the same style as the examples above.\n")
    lines.append(f"Code:\n{code}")
    lines.append(f"Question: {question}")
    lines.append("Answer:")
    return "\n".join(lines)

def build_teacher_user(code, question, first):
    return f"Code:\n{code}\n\nQuestion:\n{question}\n\nStudent's first answer:\n{first}"

def build_student_revised_user(code, question, first, teacher_analysis):
    return (
        f"Code:\n{code}\n\nQuestion:\n{question}\n\n"
        f"Your first answer:\n{first}\n\nTutor's analysis and guidance:\n{teacher_analysis}\n\nImproved Answer:"
    )

def parse_teacher(text):
    labels = {
        "intent":        "QUESTION INTENT:",
        "understanding": "STUDENT UNDERSTANDING:",
        "guidance":      "GUIDANCE:",
    }
    out       = {k: "" for k in labels}
    positions = {k: text.find(v) for k, v in labels.items()}
    for k, idx in positions.items():
        if idx == -1:
            continue
        start = idx + len(labels[k])
        ends  = [p for p in positions.values() if p > idx]
        end   = min(ends) if ends else len(text)
        out[k] = text[start:end].strip()
    return out

print("✅ ToM-SLM prompts defined.")

In [ ]:
# ── Step 1: Student generates a first answer
student_tok, student_model = load_model(STUDENT_MODEL_ID, "Student")

first_answer = run_model(
    student_model, student_tok,
    NOINTV_SYSTEM,
    build_nointv_user(code, question),
    MAX_TOKENS_FIRST,
)
print("Student first answer:")
print("  ", first_answer)

offload(student_model, "Student")

In [ ]:
# ── Step 2: Teacher-SLM builds a ToM diagnosis
teacher_slm_tok, teacher_slm_model = load_model(TEACHER_SLM_MODEL_ID, "Teacher-SLM")

teacher_raw = run_model(
    teacher_slm_model, teacher_slm_tok,
    TEACHER_SYSTEM,
    build_teacher_user(code, question, first_answer),
    MAX_TOKENS_TEACHER,
)
parsed = parse_teacher(teacher_raw)
print("QUESTION INTENT:\n ", parsed["intent"])
print("\nSTUDENT UNDERSTANDING:\n ", parsed["understanding"])
print("\nGUIDANCE:\n ", parsed["guidance"])

offload(teacher_slm_model, "Teacher-SLM")
del teacher_slm_model
gc.collect()

In [ ]:
# ── Step 3: Student revises using Teacher-SLM guidance
restore(student_model, "Student")

revised_answer = run_model(
    student_model, student_tok,
    STUDENT_REVISED_SYSTEM,
    build_student_revised_user(code, question, first_answer, teacher_raw),
    MAX_TOKENS_REVISED,
)
print("Revised answer (ToM-SLM):")
print("  ", revised_answer)

offload(student_model, "Student")

In [ ]:
# ── Step 4: Judge scores the revised answer
del student_model
gc.collect()

judge_tok, judge_model = load_model(JUDGE_MODEL_ID, "Judge")
scores = run_judge(judge_model, judge_tok, code, question, reference, revised_answer)
show_scores(scores, "ToM-SLM")

offload(judge_model, "Judge")
del judge_model
gc.collect()